# Custom Evaluation with LlamaIndex
In this notebook, we evaluate 3 different embedding models:

1. proprietary OpenAI embedding,
2. open source `BAAI/bge-small-en`, and
3. our finetuned embedding model.

We consider 2 evaluation approaches:

1. a simple custom **hit rate** metric
2. using `InformationRetrievalEvaluator` from sentence_transformers

We show that finetuning on synthetic (LLM-generated) dataset significantly improve upon an opensource embedding model.

In [1]:
import json
from tqdm.notebook import tqdm
import pandas as pd

## Load Data

First, let's load the synthetic dataset we automatically generated from our corpus (without having access to any labellers).



In [2]:
TRAIN_DATASET_PATH = '/kaggle/input/split-dataset-1500/train_dataset.json'
TEST_DATASET_PATH = '/kaggle/input/split-dataset-1500/test_dataset.json'

In [3]:
with open(TRAIN_DATASET_PATH, 'r') as f:
    train_dataset = json.load(f)

with open(TEST_DATASET_PATH, 'r') as f:
    test_dataset = json.load(f)

## Define eval function
**Option 1**: We use a simple hit rate metric for evaluation:

* for each (query, relevant_doc) pair,
* we retrieve top-k documents with the query, and
* it's a **hit** if the results contain the relevant_doc.

This approach is very simple and intuitive, and we can apply it to both the proprietary OpenAI embedding as well as our open source and fine-tuned embedding models.

### I'm not evaluating using hit rate, due to llama-index dependency issues

In [4]:
# def evaluate(
#     dataset,
#     embed_model,
#     top_k=5,
#     verbose=False,
# ):
#     corpus = dataset['corpus']
#     queries = dataset['queries']
#     relevant_docs = dataset['relevant_docs']

#     service_context = ServiceContext.from_defaults(embed_model=embed_model)
#     nodes = [TextNode(id_=id_, text=text) for id_, text in corpus.items()] 
#     index = VectorStoreIndex(
#         nodes, 
#         service_context=service_context, 
#         show_progress=True
#     )
#     retriever = index.as_retriever(similarity_top_k=top_k)

#     eval_results = []
#     for query_id, query in tqdm(queries.items()):
#         retrieved_nodes = retriever.retrieve(query)
#         retrieved_ids = [node.node.node_id for node in retrieved_nodes]
#         expected_id = relevant_docs[query_id][0]
#         is_hit = expected_id in retrieved_ids  # assume 1 relevant doc
        
#         eval_result = {
#             'is_hit': is_hit,
#             'retrieved': retrieved_ids,
#             'expected': expected_id,
#             'query': query_id,
#         }
#         eval_results.append(eval_result)
#     return eval_results

**Option 2**: We use the `InformationRetrievalEvaluator` from sentence_transformers.

This provides a more comprehensive suite of metrics, but we can only run it against the sentencetransformers compatible models (open source and our finetuned model, not the OpenAI embedding model).

In [5]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 6.3 MB/s eta 0:00:00


In [6]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import SentenceTransformer

def evaluate_st(
    dataset,
    model_id,
    name,
):
    corpus = dataset['corpus']
    queries = dataset['queries']
    relevant_docs = dataset['relevant_docs']

    evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_docs, name=name)
    model = SentenceTransformer(model_id)
    return evaluator(model, output_path='results/')

## Run Evals
### OpenAI
Note: this might take a few minutes to run since we have to embed the corpus and queries

In [7]:
# ada = OpenAIEmbedding()
# ada_val_results = evaluate(val_dataset, ada)

In [8]:
# df_ada = pd.DataFrame(ada_val_results)

In [9]:
# hit_rate_ada = df_ada['is_hit'].mean()
# hit_rate_ada

## BAAI/bge-small-en


In [10]:
bge = "local:BAAI/bge-small-en"
# bge_val_results = evaluate(val_dataset, bge)

In [11]:
# df_bge = pd.DataFrame(bge_val_results)


In [12]:
# hit_rate_bge = df_bge['is_hit'].mean()
# hit_rate_bge

In [13]:
!mkdir results

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [14]:
evaluate_st(test_dataset, "BAAI/bge-small-en", name='bge')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.8045203183439374

## Finetuned

In [15]:
# finetuned = "local:exp_finetune"
# val_results_finetuned = evaluate(val_dataset, finetuned)

In [16]:
# df_finetuned = pd.DataFrame(val_results_finetuned)

In [17]:
# hit_rate_finetuned = df_finetuned['is_hit'].mean()
# hit_rate_finetuned

In [18]:
finetuned_path = "/kaggle/input/finetuned-embeddings-model/exp_finetune" 

In [19]:
evaluate_st(test_dataset, finetuned_path, name='finetuned')


0.8761822821557335

## Summary of Results
### Hit rate

In [20]:

# df_ada['model'] = 'ada'
# df_bge['model'] = 'bge'
# df_finetuned['model'] = 'fine_tuned'



We can see that fine-tuning our small open-source embedding model drastically improve its retrieval quality (even approaching the quality of the proprietary OpenAI embedding)!


In [21]:
# df_all = pd.concat([df_ada, df_bge, df_finetuned])
# df_all.groupby('model').mean('is_hit')

## InformationRetrievalEvaluator


In [22]:
df_st_bge = pd.read_csv('results/Information-Retrieval_evaluation_bge_results.csv')
df_st_finetuned = pd.read_csv('results/Information-Retrieval_evaluation_finetuned_results.csv')


We can see that embedding finetuning improves metrics consistently across the suite of eval metrics



In [23]:
df_st_bge['model'] = 'bge'
df_st_finetuned['model'] = 'fine_tuned'
df_st_all = pd.concat([df_st_bge, df_st_finetuned])
df_st_all = df_st_all.set_index('model')
df_st_all

,epoch,steps,cos_sim-Accuracy@1,cos_sim-Accuracy@3,cos_sim-Accuracy@5,cos_sim-Accuracy@10,cos_sim-Precision@1,cos_sim-Recall@1,cos_sim-Precision@3,cos_sim-Recall@3,...,dot_score-Recall@1,dot_score-Precision@3,dot_score-Recall@3,dot_score-Precision@5,dot_score-Recall@5,dot_score-Precision@10,dot_score-Recall@10,dot_score-MRR@10,dot_score-NDCG@10,dot_score-MAP@100
model,,,,,,,,,,,,,,,,,,,,,
bge,-1,-1,0.721239,0.871681,0.929204,0.955752,0.721239,0.721239,0.290560,0.871681,...,0.721239,0.290560,0.871681,0.185841,0.929204,0.095575,0.955752,0.802788,0.840372,0.804520
fine_tuned,-1,-1,0.809735,0.929204,0.946903,0.969027,0.809735,0.809735,0.309735,0.929204,...,0.809735,0.309735,0.929204,0.189381,0.946903,0.096903,0.969027,0.875323,0.898714,0.876182
